In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
columns = ['conflict_id', 'year_mo', 'year_mo_numeric', 'start_date', 'start_date_numeric', 'end_date', 'end_date_numeric', 'conflict_age', 'real_observation',
           'best', 'log_best', 'isocode', 'country', 'agreement','first_agreement_date', 'treated_agreement']
df = pd.read_csv('../../../data/output/conflict_level/conflict_panel.csv', low_memory=False, usecols=columns)
df

,conflict_id,year_mo,best,isocode,country,log_best,agreement,start_date,end_date,real_observation,year_mo_numeric,start_date_numeric,end_date_numeric,conflict_age,first_agreement_date,treated_agreement
0,205,1989-01,0.0,IRN,Iran,0.000000,0,1990-04,2022-11,0,1,16,407,NaN,0,0
1,205,1989-02,0.0,IRN,Iran,0.000000,0,1990-04,2022-11,0,2,16,407,NaN,0,0
2,205,1989-03,0.0,IRN,Iran,0.000000,0,1990-04,2022-11,0,3,16,407,NaN,0,0
3,205,1989-04,0.0,IRN,Iran,0.000000,0,1990-04,2022-11,0,4,16,407,NaN,0,0
4,205,1989-05,0.0,IRN,Iran,0.000000,0,1990-04,2022-11,0,5,16,407,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08,0.0,SOM,Somalia,0.000000,0,2024-12,2024-12,0,428,432,432,NaN,0,0
86828,16379,2024-09,0.0,SOM,Somalia,0.000000,0,2024-12,2024-12,0,429,432,432,NaN,0,0
86829,16379,2024-10,0.0,SOM,Somalia,0.000000,0,2024-12,2024-12,0,430,432,432,NaN,0,0
86830,16379,2024-11,0.0,SOM,Somalia,0.000000,0,2024-12,2024-12,0,431,432,432,NaN,0,0


### **Redefine treatment**

We will consider the first agreement as the treatment, and the condition to be 1 is that in the previous 6 months (including the month of the agreement) the sum of fatalities is greater than 12.

This is a common threshold used in the UCDP dataset to define active conflicts.

In [12]:
df = df.sort_values(["conflict_id", "year_mo"]).copy()

df["pre6_best_sum"] = (
    df.groupby("conflict_id")["best"]
      .transform(lambda s: s.rolling(window=7, min_periods=1).sum())
)


df["first_agreement"] = (
    (df["agreement"].eq(1)) &
    (df.groupby("conflict_id")["agreement"].cumsum().eq(1))
).astype("int8")

df["first_agreement_active"] = (
    df["first_agreement"] &
    df["pre6_best_sum"].ge(12)
).astype("int8")

In [13]:
before = df["first_agreement"].sum()
after = df["first_agreement_active"].sum()

conflicts_before = df.groupby("conflict_id")["first_agreement"].max().sum()
conflicts_after = df.groupby("conflict_id")["first_agreement_active"].max().sum()

print("Obs before:", before)
print("Obs after:", after)
print("Conflicts before:", conflicts_before)
print("Conflicts after:", conflicts_after)
print("Dropped conflicts:", conflicts_before - conflicts_after)

Obs before: 71
Obs after: 64
Conflicts before: 71
Conflicts after: 64
Dropped conflicts: 7


In [14]:
df[df.first_agreement_active==1]['conflict_id'].unique()

array([  209,   218,   221,   230,   233,   234,   251,   267,   269,
         283,   288,   289,   299,   300,   308,   309,   314,   316,
         327,   330,   332,   333,   336,   337,   341,   352,   366,
         369,   374,   382,   384,   385,   386,   388,   389,   390,
         392,   393,   394,   395,   397,   398,   401,   402,   403,
         404,   408,   409,   410,   412,   416,   417,   419,   421,
         423,   426, 11344, 11345, 11346, 11347, 13243, 13306, 13324,
       14275])

222 329 not in the montly filter

In [15]:
df.loc[df['conflict_id']==222][['pre6_best_sum', 'conflict_id', 'year_mo', 'best', 'first_agreement']]

,pre6_best_sum,conflict_id,year_mo,best,first_agreement
2160,0.000000,222,1989-01,0.0,0
2161,0.000000,222,1989-02,0.0,0
2162,0.000000,222,1989-03,0.0,0
2163,0.000000,222,1989-04,0.0,0
2164,0.000000,222,1989-05,0.0,0
...,...,...,...,...,...
2587,671.666667,222,2024-08,113.0,0
2588,706.833333,222,2024-09,101.0,0
2589,781.000000,222,2024-10,161.0,0
2590,843.000000,222,2024-11,111.0,0


In [16]:
treated = df.loc[df["first_agreement"] == 1, ["conflict_id", "year_mo", "pre6_best_sum", 'country']].copy()
treated["status"] = np.where(treated["pre6_best_sum"] >= 12, "Active", "Inactive")
def summary_stats(x):
    return pd.Series({
        "N": x.shape[0],
        "Mean": x.mean(),
        "Median": x.median(),
        "SD": x.std()
    })

summary_all = summary_stats(treated["pre6_best_sum"])
summary_active = summary_stats(treated.loc[treated["status"] == "Active", "pre6_best_sum"])
summary_inactive = summary_stats(treated.loc[treated["status"] == "Inactive", "pre6_best_sum"])
bins = [1, 12, 25, 100, 500, 1000, np.inf]
labels = ["1–11 deaths", "12–24 deaths", "25–99 deaths",
          "100–499 deaths", "500–999 deaths", "1,000+ deaths"]

treated["death_bin"] = pd.cut(
    treated["pre6_best_sum"],
    bins=bins,
    labels=labels,
    right=False
)
dist_all = treated["death_bin"].value_counts().reindex(labels, fill_value=0)
dist_active = treated.loc[treated["status"] == "Active", "death_bin"].value_counts().reindex(labels, fill_value=0)
dist_inactive = treated.loc[treated["status"] == "Inactive", "death_bin"].value_counts().reindex(labels, fill_value=0)
table = pd.DataFrame({
    "All": [
        summary_all["Mean"],
        summary_all["Median"],
        summary_all["SD"],
        *dist_all.tolist()
    ],
    "Active": [
        summary_active["Mean"],
        summary_active["Median"],
        summary_active["SD"],
        *dist_active.tolist()
    ],
    "Inactive": [
        summary_inactive["Mean"],
        summary_inactive["Median"],
        summary_inactive["SD"],
        *dist_inactive.tolist()
    ]
}, index=[
    "Mean", "Median", "SD",
    "1–11 deaths", "12–24 deaths", "25–99 deaths",
    "100–499 deaths", "500–999 deaths", "1,000+ deaths"
])

print(table)

                         All        Active  Inactive
Mean             3745.843815   4154.904858  5.857143
Median            228.666667    247.041667  7.000000
SD              20362.584793  21423.847576  2.853569
1–11 deaths         7.000000      0.000000  7.000000
12–24 deaths        4.000000      4.000000  0.000000
25–99 deaths       15.000000     15.000000  0.000000
100–499 deaths     22.000000     22.000000  0.000000
500–999 deaths     10.000000     10.000000  0.000000
1,000+ deaths      13.000000     13.000000  0.000000


### Who are the inactive conflicts?

In [ ]:
treated[treated['status']=='Inactive'][['conflict_id', 'country', 'year_mo', 'pre6_best_sum']].sort_values('pre6_best_sum', ascending=False).head(10)

### Keep real observations for control units.

In [ ]:
df = df.loc[(df['real_observation']==1) | (df['first_agreement_active']==1)].copy()
df

## Construct gvar

In [ ]:
gvar_by_conflict = df.groupby('conflict_id').agg(
    gvar = ('year_mo_numeric', lambda x: x[df['first_agreement_active'] == 1].min() if (df['first_agreement_active'] == 1).any() else 0)
).fillna(0)
df = df.join(gvar_by_conflict, on="conflict_id", how="left")
df

## Construct pseudo treatment for never treated units

In [ ]:
treated_g = (
    df
    .filter(pl.col("gvar") > 0)
    .select("conflict_id", "gvar")
    .unique()
    .sort("conflict_id")
    .with_row_index("draw_id", offset=1)
)


controls = (
    df
    .select("conflict_id", "gvar")
    .unique()
    .filter(pl.col("gvar") == 0)
)

rng = np.random.default_rng(20240315)

controls = controls.with_columns(
    pl.Series(
        "draw_id",
        rng.integers(1, treated_g.height + 1, size=controls.height)
    )
)

controls = (
    controls
    .join(
        treated_g.select(
            "draw_id",
            pl.col("gvar").alias("pseudo_gvar")
        ),
        on="draw_id",
        how="left"
    )
    .select("conflict_id", "pseudo_gvar")
)

df = df.join(controls, on="conflict_id", how="left")



## Construct the relative time to treatment

In [ ]:
df = df.with_columns(
    pl.when(pl.col("gvar") > 0)
    .then(pl.col("year_mo_numeric") - pl.col("gvar"))
    .otherwise(pl.col("year_mo_numeric") - pl.col("pseudo_gvar"))
    .alias("event_time_for_covs")
)


## Construct pre-violence level

In [ ]:

pre_violence = (
    df
    .filter(
        (pl.col("event_time_for_covs") >= -4) &
        (pl.col("event_time_for_covs") <= -2)
    )
    .group_by("conflict_id")
    .agg(
        pl.col("log_best").mean().alias("pre_violence_level")
    )
)
fallback_mean = (
    df
    .group_by("conflict_id")
    .agg(
        pl.col("log_best").mean().alias("mean_log_best")
    )
)
df = (
    df
    .join(pre_violence, on="conflict_id", how="left")
    .join(fallback_mean, on="conflict_id", how="left")
    .with_columns(
        pl.coalesce(
            pl.col("pre_violence_level"),
            pl.col("mean_log_best")
        ).alias("pre_violence_level")
    )
    .drop("mean_log_best")
)

## Construct age_at_treatment

In [ ]:
age_at_treat = (
    df
    .filter(
        ((pl.col("gvar") > 0) & (pl.col("year_mo_numeric") == pl.col("gvar"))) |
        ((pl.col("gvar") == 0) & (pl.col("year_mo_numeric") == pl.col("pseudo_gvar")))
    )
    .select("conflict_id", pl.col("conflict_age").alias("age_at_treatment"))
    .unique("conflict_id")
)
df = (
    df
    .join(age_at_treat, on="conflict_id", how="left")
    .with_columns(
        pl.col("age_at_treatment").fill_null(0)
    )
)